# 05. 실시간 엔드포인트 배포

## 이 노트북에서 배우는 것
- 최적 모델(튜닝 결과)로 **실시간 추론 엔드포인트** 생성
- 요청/응답 직렬화기(serializer/deserializer) 설정
- 배포된 엔드포인트로 **스모크 테스트**

> ⚠️ **비용 주의**: 실시간 엔드포인트는 삭제 전까지 **시간당 과금**됩니다.
> 워크샵이 끝나면 `06_inference.ipynb` 의 정리(cleanup) 셀을 반드시 실행하세요.

`04_tuning.ipynb` 를 먼저 실행했어야 합니다.

## 0. 환경 복원 & 모델 재구성

튜닝에서 고른 최적 모델 아티팩트로 배포용 `Model` 객체를 만듭니다.

In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.model import Model

sess = sagemaker.Session()
role = get_execution_role()

%store -r bucket prefix xgb_image_uri best_model_data

model = Model(
    image_uri=xgb_image_uri,
    model_data=best_model_data,
    role=role,
    sagemaker_session=sess,
)
print("model artifact:", best_model_data)

## 1. 엔드포인트 배포

배포에는 수 분이 걸립니다. CSV로 요청/응답을 주고받도록 직렬화기를 지정합니다.

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

endpoint_name = "student-success-endpoint"

predictor = model.deploy(
    initial_instance_count=1,
    # TODO: 실시간 추론용 인스턴스 타입을 지정하세요 (예: "ml.m5.large").
    instance_type=____,
    endpoint_name=endpoint_name,
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer(),
)
print("deployed:", endpoint_name)

## 2. 스모크 테스트

테스트셋에서 한 샘플을 꺼내 엔드포인트에 보내 봅니다.

In [ ]:
import numpy as np
import pandas as pd
from sagemaker.s3 import S3Downloader

%store -r test_x_s3
S3Downloader.download(test_x_s3, "smoke")
X = pd.read_csv("smoke/test_x.csv", header=None)
sample = X.iloc[0].tolist()

# CSVSerializer가 리스트를 CSV 한 줄로 변환해 전송합니다.
resp = predictor.predict(sample)          # -> [['0.1', '0.2', '0.7']]
proba = [float(p) for p in resp[0]]
classes = ["Dropout", "Enrolled", "Graduate"]

# TODO: 확률이 가장 큰 클래스를 예측 라벨로 고르세요. (힌트: int(np.argmax(proba)))
pred_idx = ____
print("probabilities:", dict(zip(classes, [round(p, 3) for p in proba])))
print("prediction   :", classes[pred_idx])

## 3. 엔드포인트 이름 저장

In [ ]:
%store endpoint_name
print("stored endpoint_name =", endpoint_name)

> 🔒 **보안 메모**: SageMaker 엔드포인트는 공개(public) URL이 아니라 **AWS IAM/SigV4 서명**으로만
> 호출할 수 있는 프라이빗 엔드포인트입니다. 뒤의 웹앱(`07`)도 IAM 자격증명으로 호출합니다.

## 참고: 프로덕션 배포 전략 (Blue/Green · Rolling)

위에서는 `model.deploy()` 로 간단히 배포했지만, **프로덕션 환경에서 모델을 교체**할 때는
무중단 배포와 자동 롤백이 필요합니다. SageMaker는 `UpdateEndpoint` 시 `DeploymentConfig` 로
**블루/그린(Blue/Green)** 또는 **롤링(Rolling)** 배포 전략을 지원합니다.

아래는 참고용 코드 예시입니다 (이 워크샵에서는 실행하지 않습니다).

```python
import boto3

sm = boto3.client("sagemaker")

# 블루/그린 배포: 새 모델로 트래픽의 일부를 보내고, 이상 없으면 전환
sm.update_endpoint(
    EndpointName="student-success-endpoint",
    EndpointConfigName="new-endpoint-config",
    DeploymentConfig={
        "BlueGreenUpdatePolicy": {
            "TrafficRoutingConfiguration": {
                "Type": "CANARY",                  # CANARY 또는 ALL_AT_ONCE
                "CanarySize": {"Type": "INSTANCE_COUNT", "Value": 1},
                "WaitIntervalInSeconds": 300,      # 카나리 관찰 시간 (5분)
            },
            "TerminationWaitInSeconds": 120,        # 전환 후 구 인스턴스 유지 시간
            "MaximumExecutionTimeoutInSeconds": 1800,
        },
        "AutoRollbackConfiguration": {
            "Alarms": [{"AlarmName": "HighLatencyAlarm"}]  # CloudWatch 알람 기반 자동 롤백
        },
    },
)
```

| 전략 | 특징 | 적합한 경우 |
|------|------|------------|
| Blue/Green (Canary) | 새 모델에 트래픽 일부만 먼저 전송, 알람 없으면 전환 | 안전성 최우선 |
| Blue/Green (All-at-once) | 새 인스턴스 준비 후 한 번에 전환 | 빠른 교체 |
| Rolling | 인스턴스를 하나씩 교체 | 대규모 엔드포인트에서 점진적 교체 |

> 자세한 내용: https://docs.aws.amazon.com/sagemaker/latest/dg/deployment-guardrails.html

✅ **완료!** 엔드포인트가 살아 있습니다. 다음은 `06_inference.ipynb` — 실시간 추론과 **리소스 정리**.